# 02 — Detection results (efficacy demo)

Reproduce the main test-set curves from compact saved scores.
**No GPU / no raw EEG required.**

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
)
from IPython.display import Image, display

ROOT = Path("..").resolve()
METRICS = ROOT / "docs" / "showcase" / "metrics"
FIGS = ROOT / "docs" / "showcase" / "figures"

data = np.load(METRICS / "test_scores_smooth15.npz")
y, p = data["y_full"], data["p_full"]
prev = float(data["prevalence"])
print(f"Windows: {int(data['n_windows']):,}")
print(f"Prevalence: {prev:.4f}")
print(f"PR-AUC: {average_precision_score(y, p):.4f}")
print(f"ROC-AUC: {roc_auc_score(y, p):.4f}")

## Why PR-AUC matters more than accuracy here

Seizures are rare (~3.8% of test windows). A dummy model that always predicts "no seizure" gets **~96% accuracy** and is useless.
PR-AUC asks: when the model ranks windows by risk, how well do true seizures rise to the top?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
PrecisionRecallDisplay.from_predictions(y, p, ax=axes[0], name="Conformer smooth")
axes[0].axhline(prev, ls="--", color="gray", label="chance")
axes[0].legend(loc="upper right")
axes[0].set_title("Precision–Recall")
RocCurveDisplay.from_predictions(y, p, ax=axes[1], name="Conformer smooth")
axes[1].plot([0, 1], [0, 1], ls="--", color="gray")
axes[1].set_title("ROC")
fig.tight_layout()
plt.show()

## Improvement over classical baselines

Soundbite: Conformer + smoothing beats bandpower logistic regression by **+0.38 PR-AUC** and EEGNet by **+0.29 PR-AUC**.

In [ ]:
lr = json.loads((METRICS / "spectral_lr_test.json").read_text())
baseline = json.loads((METRICS / "baseline_test.json").read_text())
raw = json.loads((METRICS / "conformer_ft_test_raw.json").read_text())
smooth = json.loads((METRICS / "conformer_ft_test_smooth15.json").read_text())
rows = [
    ("Bandpower logistic regression", lr),
    ("EEGNet CNN", baseline),
    ("Conformer raw", raw),
    ("Conformer + 15s max-smooth", smooth),
]
print(f"{'model':34s} {'PR-AUC':>8s} {'ROC-AUC':>8s}")
for name, m in rows:
    print(f"{name:34s} {m['pr_auc']:8.3f} {m['roc_auc']:8.3f}")
print(f"\nConformer+smooth vs LogReg: +{smooth['pr_auc'] - lr['pr_auc']:.3f} PR-AUC")
print(f"Conformer+smooth vs EEGNet:  +{smooth['pr_auc'] - baseline['pr_auc']:.3f} PR-AUC")

display(Image(filename=str(FIGS / "model_comparison.png")))

## Precision at fixed recall targets

"Catch 50% / 70% / 90% of seizure windows — how precise are we?"

In [ ]:
precision, recall, thresholds = precision_recall_curve(y, p)

def precision_at_recall(target: float) -> float:
    # highest threshold that still achieves >= target recall
    ok = np.where(recall[:-1] >= target)[0]
    if len(ok) == 0:
        return float("nan")
    i = ok[-1]
    return float(precision[i])

for t in (0.5, 0.7, 0.9):
    print(f"Precision @ recall {t:.0%}: {precision_at_recall(t):.3f}")

display(Image(filename=str(FIGS / "operating_points.png")))

## Takeaway

The Conformer + smoothing stack is a clear win over the starter EEGNet on ranking quality (PR-AUC **0.17 → 0.46** on test).
That is strong for a **portfolio / research demo**. Clinical deployment still needs the reality check in notebook 03.